In [ ]:
#LOADING REQUIRED LIBRARIES

#Data Visualization libraries
import pandas as pd
import seaborn as sns
import numpy as np
import os
import matplotlib.pyplot as plt
%matplotlib inline
from itertools import product

#For proper display of all columns
from IPython.display import display
pd.options.display.max_columns = None

#Import warnings
import warnings
warnings.filterwarnings("ignore")
import gc
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

In [ ]:
# Loading the datasets
items = pd.read_csv("items.csv")
shops = pd.read_csv("shops.csv")
cats = pd.read_csv("item_categories.csv")
train = pd.read_csv("sales_train.csv")

# Creating a data dictionary for later usage
train_data_dict = {"items": items,
                   "shops": shops, 
                   "cats": cats,
                   "train": train}

# Setting index to ID to avoid droping it later
test = pd.read_csv("test.csv").set_index("ID")


In [ ]:
# Taking unique shop and item ids
test_shop_ids = test["shop_id"].unique()
test_item_ids = test["item_id"].unique()

print("Data set size total:", train.shape[0])

# Only shops that exist in test set.
train = train[train["shop_id"].isin(test_shop_ids)]
# Only items that exist in test set.
train = train[train["item_id"].isin(test_item_ids)]

print("Data set size leaking in test", train.shape[0])


# Data Pre-Processing

**Missing value count:**

In [ ]:
# Lets check the missing
for d_name, d in train_data_dict.items():
    print(
        "\n===={} data missing value count:====\n{}".format(
            d_name.upper(), missing_val_check(d)
        )
    )

**No missing values found in the data. Lets check for outliers:**

In [ ]:
# Lets check the missing
for d_name, d in train_data_dict.items():
    print(
        "\n===={} data statistical description:====\n{}".format(
            d_name.upper(), d.describe()
        )
    )

In [ ]:
# Outlier and anomoly fix
train = train.query("item_price>0 and item_price<40000 and item_cnt_day<500")

We can implute this negetive value with the median value of the shop and item

**Negetive values and outliers are fixed on the train data to this stage. Now lets look at other supplimental datasets as well**

In [ ]:
print(shops.head())
print("\n\n", cats.head())
print("\n\n", items.head())

# I saw the data hence for the clean code checking if total row count = total unique item names
print(
    "\n\n Note: Total rows in items data {} and total unique item_names {}".format(
        items.shape[0], len(items["item_name"].unique())
    )
)

**Cleaning supplement data to use in our model:**

In [ ]:
# Splitting shop name to get the city name
shops["city"] = shops["shop_name"].str.split(" ").map(lambda x: x[0])
shops["city"] = shops["city"].replace("!", "", regex=True)

# Label encoding so that it is more readable
shops["city_code"] = LabelEncoder().fit_transform(shops["city"])
shops.drop(["city"], axis=1, inplace=True)
shops.drop(["shop_name"], axis=1, inplace=True)

# Splitting category name to get the item type.
cats["item_cat_split"] = cats["item_category_name"].str.split("-")
cats["type"] = cats["item_cat_split"].map(lambda x: x[0].strip())
cats["type_code"] = LabelEncoder().fit_transform(cats["type"])

# Splitting category name to get the item type.
cats["subcat"] = cats["item_cat_split"].map(
    lambda x: x[1].strip() if len(x) > 1 else x[0].strip()
)
cats["subcat_code"] = LabelEncoder().fit_transform(cats["subcat"])
cats = cats[["item_category_id", "type_code", "subcat_code"]]

# Dropping item names from items df as we discussed
# Create the date the product was first sold as a feature
items["first_sale_date"] = train.groupby("item_id").agg({"date_block_num": "min"})[
    "date_block_num"
]
# Refine null values in items table
items = items.apply(lambda x: x.fillna(x.median()) if x.dtype != "O" else x, axis=0)
items.drop(["item_name"], axis=1, inplace=True)

gc.collect()

**Merging suppliment data to train and test**

In [ ]:
train = (
    train.merge(items, on="item_id", how="left")
    .merge(shops, on="shop_id", how="left")
    .merge(cats, on="item_category_id", how="left")
)

test = (
    test.merge(items, on="item_id", how="left")
    .merge(shops, on="shop_id", how="left")
    .merge(cats, on="item_category_id", how="left")
)

**Lets quickly look at our test data and see what additional item shop pairs we have.**

In [ ]:
# Creating a test dummy so that we dont make any changes in test data
test_dummy = test.copy()
test_dummy["shp_itm"] = (
    test_dummy["shop_id"].astype(str) + "_" + test_dummy["item_id"].astype(str)
)
test_shp_itm_ls = list(test_dummy["shp_itm"].unique())

train["shp_itm"] = train["shop_id"].astype(str) + "_" + train["item_id"].astype(str)
trn_shp_itm_ls = list(train["shp_itm"].unique())

print("Total unique shop and item combination in the train data :", len(trn_shp_itm_ls))
print("Total unique shop and item combination in the test data:", len(test_shp_itm_ls))
print(
    "Any new shop item sales needs to be predicted:",
    str(len(list(set(test_shp_itm_ls) - set(trn_shp_itm_ls))) > 0),
)
print(
    "Total new shop item combination in the test data",
    str(len(list(set(test_shp_itm_ls) - set(trn_shp_itm_ls)))),
)

**Grouping on month level as our we need to predict next month total sales**

In [ ]:
# Group by month in this case "date_block_num" and aggregate features.
train_monthly = train.groupby(
    [
        "date_block_num",
        "shop_id",
        "item_category_id",
        "item_id",
        "city_code",
        "type_code",
        "subcat_code",
    ],
    as_index=False,
)
train_monthly = train_monthly.agg(
    {"item_price": "mean", "item_cnt_day": ["sum", "mean", "count"]}
).reset_index(drop="True")
train_monthly.columns = [
    "date_block_num",
    "shop_id",
    "item_category_id",
    "item_id",
    "city_code",
    "type_code",
    "subcat_code",
    "item_price",
    "item_cnt",
    "mean_item_cnt",
    "transactions",
]

**Note**: I tried the below approach but due to memory/time constraints I decide to remove this approach. 
Also it was not giving a significant improvement in model performance. 

In [ ]:
test["item_cnt"] = 0
test["date_block_num"] = 34
cols = [
    "shop_id",
    "item_id",
    "date_block_num",
    "item_category_id",
    "city_code",
    "type_code",
    "subcat_code",
    "first_sale_date",
]
train_monthly = pd.concat(
    [train_monthly, test], ignore_index=True, sort=False, keys=cols
)

**Now lets roll this data to monthly level for analysing trends**

In [ ]:
# Extract time based features.
train_monthly["year"] = train_monthly["date_block_num"].apply(
    lambda x: ((x // 12) + 2013)
)
train_monthly["month"] = train_monthly["date_block_num"].apply(lambda x: (x % 12))

# Feature Engineering

We will traditionally create lead, lag features in a time series based problems + some intuitive features which might add some value to our model. But we need to keep in mind we dont create any leaky feature which led to overfit.

**Price is a key lever for the sales, we can create some features which might help the model to capture the trend to predict the future sales**
1. Creating price per item feature
2. Creating price difference from the current price - the historical min item price
3. Creating first selling month of a product and a boolean to check if a product is newly launched product at that instance
4. Creating rolling window features
5. Creating min and max price of that product
6. Creating lag features
7. Creating trend features


**Intuitive features**

In [ ]:
# Adding min and max price of the item
item_price = (
    train_monthly.sort_values("date_block_num")
    .groupby(["item_id"], as_index=False)
    .agg({"item_price": [np.min, np.max]})
)
item_price.columns = ["item_id", "hist_min_item_price", "hist_max_item_price"]
train_monthly = pd.merge(train_monthly, item_price, on="item_id", how="left")

# Adding average price difference w.r.t the current month with min and max historic price
train_monthly["price_increase"] = (
    train_monthly["item_price"] - train_monthly["hist_min_item_price"]
)
train_monthly["price_decrease"] = (
    train_monthly["hist_max_item_price"] - train_monthly["item_price"]
)

# Adding first month of selling of a product
train_monthly["first_selling_date_block"] = train_monthly.groupby("item_id")[
    "date_block_num"
].min()

# Adding a boolean if a product is a newly launched product or not
train_monthly["is_new_product"] = (
    train_monthly["first_selling_date_block"] == train_monthly["date_block_num"]
)

**Rolling window features**

In [ ]:
# Rolling window features
# Min value
f_min = lambda x: x.rolling(window=3, min_periods=1).min()
# Max value
f_max = lambda x: x.rolling(window=3, min_periods=1).max()
# Mean value
f_mean = lambda x: x.rolling(window=3, min_periods=1).mean()
# Standard deviation
f_std = lambda x: x.rolling(window=3, min_periods=1).std()

func_list = [f_min, f_max, f_mean, f_std]
func_name = ["min", "max", "mean", "std"]

for i in range(len(func_list)):
    train_monthly[("item_cnt_%s" % func_name[i])] = (
        train_monthly.sort_values("date_block_num")
        .groupby(["shop_id", "item_category_id", "item_id"])["item_cnt"]
        .apply(func_list[i])
    )

# Fill the empty std features with 0
train_monthly["item_cnt_std"].fillna(0, inplace=True)

**Creating lag features**

In [ ]:
# Creating lag_features function to create the lags which helps model to predict the future month sales.

def lag_features(df, lags, col_list):
    for col_name in col_list:
        tmp = df[["date_block_num", "shop_id", "item_id", col_name]]
        for i in lags:
            shifted = tmp.copy()
            shifted.columns = [
                "date_block_num",
                "shop_id",
                "item_id",
                col_name + "_lag_" + str(i),
            ]
            shifted["date_block_num"] += i
            df = pd.merge(
                df, shifted, on=["date_block_num", "shop_id", "item_id"], how="left"
            )
    return df

In [ ]:
# Creating last 3 month average of sales
train_monthly["qmean"] = train_monthly[
    ["item_cnt_lag_1", "item_cnt_lag_2", "item_cnt_lag_3"]
].mean(skipna=True, axis=1)

train_monthly["qmean_rev"] = train_monthly[
    ["revenue_lag_1", "revenue_lag_2", "revenue_lag_3"]
].mean(skipna=True, axis=1)

In [ ]:
# Getting some lag ratios to get some kind of trend
train_monthly["item_cat_lag1_ratio"] = (
    train_monthly["item_cnt_lag_1"] / train_monthly["item_cnt_lag_2"]
)
train_monthly["item_cat_lag1_ratio"] = (
    train_monthly["item_cat_lag1_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
)

train_monthly["item_cat_lag2_ratio"] = (
    train_monthly["item_cnt_lag_2"] / train_monthly["item_cnt_lag_3"]
)
train_monthly["item_cat_lag2_ratio"] = (
    train_monthly["item_cat_lag2_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
)

# Adding revenue ratio to get additional trend features
train_monthly["rev_lag1_ratio"] = (
    train_monthly["revenue_lag_1"] / train_monthly["revenue_lag_2"]
)
train_monthly["rev_lag2_ratio"] = (
    train_monthly["revenue_lag_2"] / train_monthly["revenue_lag_3"]
)

**Additional trend variables**

**Note:**

In addition note we can use **date_block_num as our organic growth variable** since we saw during EDA then there is an increasing trend of sales and will treat it as numeric

In [ ]:
# Saving the data
train_monthly.to_pickle("train_monthly.pkl")

del grp
del items
del shops
del cats
del train
gc.collect()

# Modelling Data Prep:

In [ ]:
train_monthly = pd.read_pickle("train_monthly.pkl")

**Clipping the target between 0 and 20 as our target is given to be in the range. Hence for better prediction and reduction in the target variance this is an important step:** 

In [ ]:
# Clipping the data first as discussed
train_monthly["item_cnt"] = train_monthly["item_cnt"].clip(0, 20)

In [ ]:
# Replacing NaNs with 0 - Just a check
train_monthly = train_monthly.fillna(0)

# Remove first 5 months of data since we used lag of 6 hence that made it defective.
# Hence doesnt provide historic information for model to learn
train_monthly = train_monthly.loc[train_monthly["date_block_num"] > 5]
cat_cols = [
    "shop_id",
    "item_id",
    "item_category_id",
    "city_code",
    "type_code",
    "subcat_code",
    "month",
]

# Lightgbm has in built categorical feature encoding however require such feature as category type
train_monthly[cat_cols] = train_monthly[cat_cols].astype("category")

**Train-Val-Split**

Below is the breakup:

Train -- > 0-33 Months

Val --> 33rd Month

Test --> 34th Month

In [ ]:
# Train-Validation-Test Split - Take 1 month of data as validation and month = 33 as our test since we dont have data for month = 34
X_train = train_monthly.loc[train_monthly["date_block_num"] < 33].drop(
    ["item_cnt"], axis=1
)
y_train = train_monthly.loc[train_monthly["date_block_num"] < 33]["item_cnt"]

X_val = train_monthly.loc[train_monthly["date_block_num"] == 33].drop(
    ["item_cnt"], axis=1
)
y_val = train_monthly.loc[train_monthly["date_block_num"] == 33]["item_cnt"]

X_test = train_monthly.loc[train_monthly["date_block_num"] == 34].drop(
    ["item_cnt"], axis=1
)
y_test = train_monthly.loc[train_monthly["date_block_num"] == 34]["item_cnt"]

# Modelling

In [ ]:
final_feat = X_train.columns.tolist()

params = {
    "objective": "mse",
    "metric": "rmse",
    "num_leaves": 250,
    "learning_rate": 0.005,  # Keeping small learning rate for better convergence
    "feature_fraction": 0.75,  # Keeping around 75% to induce generalisation in the model
    "bagging_fraction": 0.75,  # Keeping around 75% to induce generalisation in the model
    "bagging_freq": 5,
    "seed": 1,
    "verbose": 1,
    "force_row_wise": True,
}

lgb_train = lgb.Dataset(X_train[final_feat], y_train, categorical_feature=None)
lgb_eval = lgb.Dataset(
    X_val[final_feat], y_val, categorical_feature=None, reference=lgb_train
)

evals_result = {}
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=(lgb_train, lgb_eval),
    feature_name=final_feat,
    categorical_feature=cat_cols,
    verbose_eval=100,
    evals_result=evals_result,
    early_stopping_rounds=100,
)

In [ ]:
lgb.plot_importance(gbm, max_num_features=30, 
                    importance_type="gain", 
                    figsize=(15, 10))

**Predictions**

In [ ]:
X_test_cp = X_test.copy()
X_test = X_test.reset_index(drop = True)

y_test = gbm.predict(X_test[list(X_train.columns)]).clip(0, 20)
X_test_cp['item_cnt_month'] = y_test

submission = test.merge(X_test_cp[['shop_id', 'item_id', 'item_cnt_month']], on = ['shop_id', 'item_id'], how = 'left')
submission = submission.fillna(0)

submission.drop(['shop_id', 'item_id'], axis = 1, inplace = True)
submission['ID'] = submission.index
submission = submission[['ID', 'item_cnt_month']]
submission.to_csv('lgbm_submission_T2.csv', index=False)

I hope you liked my approach